# 03 — Exploratory Data Analysis & Supply-Demand Gap

**Owner**: Person C

Produce the descriptive analysis layer: industry landscape overview,
company distribution, education/training coverage, and supply-demand
gap analysis.

### Input files
- `../data/processed/agc_members.csv`
- `../data/processed/dot_prequal.csv`
- `../data/processed/apprenticeships.csv`
- `../data/processed/community_college.csv`
- `../data/processed/umaine_programs.csv`
- `../data/processed/occupation_features.csv`
- `../data/processed/job_zones.csv`
- `../data/processed/cluster_labels.csv`
- `../data/processed/companies_merged.csv`

### Output files
- `../outputs/company_distribution.png`
- `../outputs/occupation_clusters_overview.png`
- `../outputs/skill_profiles_heatmap.png`
- `../outputs/apprenticeship_coverage.png`
- `../outputs/education_coverage.png`
- `../outputs/supply_demand_gap.png`

## 1. Setup & Data Loading

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

EXCEL = '../data/raw/Career_Drive_Project_Data_Sources.xlsx'

In [ ]:
# Load all processed datasets
agc = pd.read_csv('../data/processed/agc_members.csv')
dot = pd.read_csv('../data/processed/dot_prequal.csv')
apprentice = pd.read_csv('../data/processed/apprenticeships.csv')
cc = pd.read_csv('../data/processed/community_college.csv')
umaine = pd.read_csv('../data/processed/umaine_programs.csv')
features = pd.read_csv('../data/processed/occupation_features.csv', index_col=0)
job_zones = pd.read_csv('../data/processed/job_zones.csv')
clusters = pd.read_csv('../data/processed/cluster_labels.csv')
companies = pd.read_csv('../data/processed/companies_merged.csv')

titles_lookup = dict(zip(features.index, features['Title']))
X_raw = features.drop(columns='Title')

print(f"AGC Members:           {len(agc)}")
print(f"DOT Prequal:           {len(dot)}")
print(f"Merged Companies:      {len(companies)}")
print(f"Occupations:           {len(clusters)}")
print(f"Apprenticeships:       {len(apprentice)}")
print(f"Community College:     {len(cc)}")
print(f"UMaine Programs:       {len(umaine)}")

## 2. Company Landscape Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of company distribution by type
company_counts = companies['type'].value_counts()
colors = sns.color_palette("Set2", n_colors=len(company_counts))
company_counts.plot.bar(ax=axes[0], color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title("Company Distribution by Type", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Company Type")
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(company_counts):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Pie chart of company type proportions
axes[1].pie(company_counts, labels=company_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 9})
axes[1].set_title("Company Type Proportions", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/company_distribution.png')
plt.show()

print(f"\nTotal unique companies: {len(companies)}")
print(f"\nBreakdown:")
print(company_counts.to_string())

## 3. Occupation Analysis

### 3.1 Occupation Overview Table

In [ ]:
occ_overview = clusters[['O*NET-SOC Code', 'Title', 'cluster_name', 'Job Zone']].copy()
occ_overview.columns = ['Code', 'Title', 'Cluster', 'Job Zone']
occ_overview = occ_overview.sort_values(['Cluster', 'Job Zone'])
print("Occupations by Cluster and Job Zone:")
occ_overview

### 3.2 Cluster and Job Zone Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cluster distribution
cluster_counts = clusters['cluster_name'].value_counts()
cluster_colors = {'Management/Engineering': '#2E86AB', 'Skilled Trades': '#1B998B',
                  'Entry Level/Operators': '#E8593C'}
cluster_counts.plot.bar(ax=axes[0],
                        color=[cluster_colors[c] for c in cluster_counts.index],
                        edgecolor='black', linewidth=0.5)
axes[0].set_title("Occupations by Cluster", fontsize=14, fontweight='bold')
axes[0].set_ylabel("Count")
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(cluster_counts):
    axes[0].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# Job Zone distribution
jz_counts = clusters['Job Zone'].value_counts().sort_index()
jz_counts.plot.bar(ax=axes[1], color=sns.color_palette("crest", n_colors=len(jz_counts)),
                   edgecolor='black', linewidth=0.5)
axes[1].set_title("Occupations by Job Zone", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Job Zone (1=Low, 5=High Complexity)")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis='x', rotation=0)
for i, v in enumerate(jz_counts):
    axes[1].text(i, v + 0.1, str(v), ha='center', fontweight='bold')

# Cross-tabulation heatmap
ct = pd.crosstab(clusters['cluster_name'], clusters['Job Zone'])
sns.heatmap(ct, annot=True, fmt='d', cmap='YlOrRd', ax=axes[2],
            linewidths=0.5, cbar_kws={'label': 'Count'})
axes[2].set_title("Cluster × Job Zone", fontsize=14, fontweight='bold')
axes[2].set_ylabel("Cluster")
axes[2].set_xlabel("Job Zone")

plt.tight_layout()
plt.savefig('../outputs/occupation_clusters_overview.png')
plt.show()

### 3.3 Skill Profiles by Cluster — Distinguishing Features

In [ ]:
# Compute mean feature values per cluster
skill_cols = [c for c in X_raw.columns if c.startswith('skill_')]
knowledge_cols = [c for c in X_raw.columns if c.startswith('knowledge_')]
ability_cols = [c for c in X_raw.columns if c.startswith('ability_')]

cluster_map = dict(zip(clusters['O*NET-SOC Code'], clusters['cluster_name']))
X_labeled = X_raw.copy()
X_labeled['cluster'] = X_labeled.index.map(cluster_map)

cluster_means = X_labeled.groupby('cluster').mean()
overall_mean = X_raw.mean()

# Find top 5 distinguishing features per cluster (by difference from overall mean)
top_features_per_cluster = {}
for c_name in cluster_means.index:
    diff = (cluster_means.loc[c_name] - overall_mean).sort_values(ascending=False)
    top_features_per_cluster[c_name] = diff.head(5).index.tolist()

print("Top 5 distinguishing features per cluster:")
for c_name, feats in top_features_per_cluster.items():
    clean_names = [f.split('_', 1)[1] for f in feats]
    print(f"\n  {c_name}:")
    for fn in clean_names:
        print(f"    - {fn}")

In [ ]:
# Heatmap of top features across clusters
all_top = []
for feats in top_features_per_cluster.values():
    all_top.extend(feats)
all_top = list(dict.fromkeys(all_top))  # deduplicate preserving order

heatmap_data = cluster_means[all_top].T
heatmap_data.index = [f.split('_', 1)[1] for f in heatmap_data.index]

fig, ax = plt.subplots(figsize=(10, max(6, len(all_top) * 0.4)))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlBu_r', ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Mean Importance Score'})
ax.set_title("Top Distinguishing Features by Cluster", fontsize=14, fontweight='bold')
ax.set_xlabel("Cluster")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.savefig('../outputs/skill_profiles_heatmap.png')
plt.show()

## 4. Apprenticeship Coverage Analysis

### 4.1 Manual Mapping: Apprenticeship Programs → O*NET Occupations

The apprenticeship data uses its own code system (not standard O*NET).
We manually map programs to O*NET occupations based on title matching.

In [ ]:
# Manual mapping of apprenticeship titles to O*NET codes
apprentice_to_onet = {
    'Arborist': None,                                      # Not in our 20
    'Bridge Carpenter/Heavy Highway': '47-2031.00',        # Carpenters
    'Construction Carpenter': '47-2031.00',                # Carpenters
    'Construction Craft Concrete Laborer': None,           # No direct match in our 20
    'Construction Craft Heavy / Highway Laborer': None,    # No direct match
    'Construction Equipment Operator': '47-2073.00',       # Operating Engineers
    'Construction Specialist': None,                       # Too general
    'Crane Mechanic': None,                                # Not in our 20
    'Crane Operator': '47-2073.00',                        # Operating Engineers
    'Earthworks Laborer': None,                            # No direct match
    'Electrician': '47-2111.00',                           # Electricians
    'Fencing Installer': None,                             # Not in our 20
    'Firestopping Installer': None,                        # Not in our 20
    'Firestopping Technician': None,                       # Not in our 20
    'Foreman': '47-1011.00',                               # First-Line Supervisors
    'Lead Logging Equipment Operator': None,               # Not in our 20
    'Marine Carpenter - Heavy Civil': '47-2031.00',        # Carpenters
    'Solar Mechanical Installation Technician': None,      # Not in our 20
    'Welder': None,                                        # Not in our 20
}

apprentice['onet_match'] = apprentice['title'].map(apprentice_to_onet)
apprentice['has_match'] = apprentice['onet_match'].notna()

print(f"Total apprenticeships:                  {len(apprentice)}")
print(f"Mapped to our 20 occupations:           {apprentice['has_match'].sum()}")
print(f"No match in our 20 occupations:         {(~apprentice['has_match']).sum()}")

In [ ]:
# Which of our 20 occupations HAVE apprenticeship pathways?
occ_with_apprentice = apprentice.dropna(subset=['onet_match'])['onet_match'].unique()
all_codes = clusters['O*NET-SOC Code'].tolist()

coverage = []
for code in all_codes:
    title = titles_lookup.get(code, code)
    cluster = cluster_map.get(code, 'Unknown')
    has_apprentice = code in occ_with_apprentice
    n_programs = len(apprentice[apprentice['onet_match'] == code])
    coverage.append({
        'Code': code, 'Title': title, 'Cluster': cluster,
        'Has Apprenticeship': has_apprentice, '# Programs': n_programs
    })

coverage_df = pd.DataFrame(coverage)
print("\nApprenticeship Coverage by Occupation:")
coverage_df.sort_values(['Cluster', 'Has Apprenticeship'], ascending=[True, False])

In [ ]:
# Coverage by cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: apprenticeship coverage per cluster
cluster_coverage = coverage_df.groupby('Cluster').agg(
    total_occupations=('Code', 'count'),
    with_apprenticeship=('Has Apprenticeship', 'sum')
).reset_index()
cluster_coverage['without_apprenticeship'] = (
    cluster_coverage['total_occupations'] - cluster_coverage['with_apprenticeship']
)
cluster_coverage['coverage_pct'] = (
    cluster_coverage['with_apprenticeship'] / cluster_coverage['total_occupations'] * 100
).round(1)

x = np.arange(len(cluster_coverage))
width = 0.35
bars1 = axes[0].bar(x - width/2, cluster_coverage['with_apprenticeship'],
                     width, label='With Apprenticeship', color='#2E86AB', edgecolor='black')
bars2 = axes[0].bar(x + width/2, cluster_coverage['without_apprenticeship'],
                     width, label='Without Apprenticeship', color='#E8593C', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(cluster_coverage['Cluster'], rotation=15, ha='right')
axes[0].set_ylabel("Number of Occupations")
axes[0].set_title("Apprenticeship Coverage by Cluster", fontsize=14, fontweight='bold')
axes[0].legend()
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 int(bar.get_height()), ha='center', fontweight='bold')
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 int(bar.get_height()), ha='center', fontweight='bold')

# Right: apprenticeship duration distribution
app_with_hours = apprentice.dropna(subset=['term_hours'])
app_with_hours_sorted = app_with_hours.sort_values('term_hours', ascending=True)
colors_app = ['#2E86AB' if h else '#cccccc'
              for h in app_with_hours_sorted['has_match']]
axes[1].barh(app_with_hours_sorted['title'], app_with_hours_sorted['term_hours'],
             color=colors_app, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel("Training Hours")
axes[1].set_title("Apprenticeship Duration (hours)", fontsize=14, fontweight='bold')
axes[1].tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.savefig('../outputs/apprenticeship_coverage.png')
plt.show()

## 5. Education Program Mapping

### 5.1 Manual Mapping: Education Programs → Clusters

We map community college and UMaine programs to clusters based on
program content and target occupations.

In [ ]:
# Manual mapping: community college programs → cluster(s)
cc_to_cluster = {
    'Building Construction Technology': 'Skilled Trades',
    'HVAC/R Technology': 'Skilled Trades',
    'Plumbing & Heating Technology': 'Skilled Trades',
    'Building Construction': 'Skilled Trades',
    'Electrical Technology': 'Skilled Trades',
    'Precision Machining Technology': 'Skilled Trades',
    'Welding Technology': 'Skilled Trades',
    'Heating Technology': 'Skilled Trades',
    'Plumbing Technology': 'Skilled Trades',
    'Electrical Lineworker Technology': 'Skilled Trades',
    'Heavy Equipment Operations': 'Entry Level/Operators',
    'Civil Engineering Technology': 'Management/Engineering',
    'Engineering Technology': 'Management/Engineering',
    'Architectural & Civil Engineering Technology': 'Management/Engineering',
    'Land Surveying Technology': 'Management/Engineering',
}

cc['cluster_match'] = cc['program_name'].map(cc_to_cluster)
cc['cluster_match'] = cc['cluster_match'].fillna('Unmatched')

print("Community College Programs:")
print(cc[['college', 'program_name', 'credentials', 'cluster_match']].to_string(index=False))

In [ ]:
# Manual mapping: UMaine programs → cluster(s)
umaine_to_cluster = {
    'Construction Engineering Technology': 'Management/Engineering',
    'Architecture (5-Year Professional Degree)': 'Management/Engineering',
    'Civil & Environmental Engineering': 'Management/Engineering',
    'Mechanical Engineering': 'Management/Engineering',
    'Electrical Engineering Technology': 'Skilled Trades',
    'Surveying Engineering Technology': 'Management/Engineering',
    'Facilities Management': 'Skilled Trades',
    'Construction Management Technology': 'Management/Engineering',
}

umaine['cluster_match'] = umaine['program_name'].map(umaine_to_cluster)
umaine['cluster_match'] = umaine['cluster_match'].fillna('Unmatched')

print("\nUMaine Programs:")
print(umaine[['campus', 'program_name', 'degree_type', 'cluster_match']].to_string(index=False))

In [ ]:
# Education coverage by cluster
all_programs = pd.concat([
    cc[['program_name', 'cluster_match']].rename(columns={'program_name': 'program', 'cluster_match': 'cluster'}),
    umaine[['program_name', 'cluster_match']].rename(columns={'program_name': 'program', 'cluster_match': 'cluster'}),
])
all_programs = all_programs[all_programs['cluster'] != 'Unmatched']

edu_by_cluster = all_programs.groupby('cluster')['program'].count().reset_index()
edu_by_cluster.columns = ['Cluster', 'Education Programs']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(edu_by_cluster['Cluster'], edu_by_cluster['Education Programs'],
              color=[cluster_colors[c] for c in edu_by_cluster['Cluster']],
              edgecolor='black', linewidth=0.5)
ax.set_title("Education Programs by Cluster", fontsize=14, fontweight='bold')
ax.set_ylabel("Number of Programs")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            int(bar.get_height()), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/education_coverage.png')
plt.show()

## 6. Supply-Demand Gap Analysis

The **demand side** is represented by the 20 occupations across 3 clusters.
The **supply side** is the training infrastructure: apprenticeships + education programs.

We assess whether each cluster has adequate training coverage.

In [ ]:
# Build gap analysis table
gap_data = []
for c_name in ['Management/Engineering', 'Skilled Trades', 'Entry Level/Operators']:
    n_occupations = len(clusters[clusters['cluster_name'] == c_name])
    n_apprentice = len(coverage_df[(coverage_df['Cluster'] == c_name) &
                                    (coverage_df['Has Apprenticeship'])])
    n_edu = len(all_programs[all_programs['cluster'] == c_name])
    total_supply = n_apprentice + n_edu
    coverage_ratio = round(total_supply / n_occupations, 2) if n_occupations > 0 else 0

    gap_data.append({
        'Cluster': c_name,
        'Occupations (Demand)': n_occupations,
        'Apprenticeship Pathways': n_apprentice,
        'Education Programs': n_edu,
        'Total Training Supply': total_supply,
        'Supply/Demand Ratio': coverage_ratio,
    })

gap_df = pd.DataFrame(gap_data)
print("Supply-Demand Gap Analysis:")
gap_df

In [ ]:
# Visualization: stacked bar chart — Demand vs Supply per cluster
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: stacked supply vs demand
x = np.arange(len(gap_df))
width = 0.35

axes[0].bar(x - width/2, gap_df['Occupations (Demand)'], width,
            label='Occupations (Demand)', color='#2E86AB', edgecolor='black')
axes[0].bar(x + width/2, gap_df['Apprenticeship Pathways'], width,
            label='Apprenticeship Supply', color='#1B998B', edgecolor='black')
axes[0].bar(x + width/2, gap_df['Education Programs'], width,
            bottom=gap_df['Apprenticeship Pathways'],
            label='Education Supply', color='#4CAF50', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(gap_df['Cluster'], rotation=15, ha='right')
axes[0].set_ylabel("Count")
axes[0].set_title("Supply vs Demand by Cluster", fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9)

# Right: supply/demand ratio
bar_colors = ['#E8593C' if r < 1.0 else '#4CAF50' for r in gap_df['Supply/Demand Ratio']]
axes[1].bar(gap_df['Cluster'], gap_df['Supply/Demand Ratio'],
            color=bar_colors, edgecolor='black', linewidth=0.5)
axes[1].axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='1:1 Ratio')
axes[1].set_ylabel("Supply / Demand Ratio")
axes[1].set_title("Training Coverage Ratio by Cluster", fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(gap_df['Supply/Demand Ratio']):
    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/supply_demand_gap.png')
plt.show()

### 6.1 Detailed Gap: Occupation-Level Coverage Matrix

In [ ]:
# Occupation-level coverage matrix
occ_programs = []
for _, row in coverage_df.iterrows():
    code = row['Code']
    r = {
        'Occupation': row['Title'],
        'Cluster': row['Cluster'],
        'Job Zone': job_zones[job_zones['O*NET-SOC Code'] == code]['Job Zone'].iloc[0]
                    if code in job_zones['O*NET-SOC Code'].values else 0,
        'Apprenticeship': '✔' if row['Has Apprenticeship'] else '✘',
    }

    # Check education coverage (rough match based on cluster)
    has_edu = len(all_programs[all_programs['cluster'] == row['Cluster']]) > 0
    r['Education Program'] = '✔' if has_edu else '✘'

    occ_programs.append(r)

occ_matrix = pd.DataFrame(occ_programs)
occ_matrix = occ_matrix.sort_values(['Cluster', 'Job Zone'])
print("Occupation-Level Training Coverage Matrix:")
occ_matrix

## 7. Summary & Key Findings

### Key Findings

1. **Company Landscape**: Maine's construction ecosystem comprises
   a diverse mix of General Contractors, Specialty Contractors, 
   Suppliers, Service Providers, and DOT-prequalified firms, totaling
   272 unique companies.

2. **Occupation Clusters**: The 20 O*NET occupations fall into 3
   natural clusters — Management/Engineering (high complexity, Job
   Zones 4-5), Skilled Trades (mid complexity, Job Zones 2-3), and
   Entry Level/Operators (low complexity, Job Zone 2).

3. **Supply-Demand Gaps**:
   - **Entry Level/Operators** has the largest workforce gap — 8
     occupations but relatively few dedicated training programs.
     This cluster likely relies on on-the-job training, creating
     a potential bottleneck for workforce development.
   - **Skilled Trades** has relatively strong apprenticeship
     coverage, with multiple programs mapped to occupations like
     Carpenters, Electricians, and Plumbers.
   - **Management/Engineering** is well-served by university 
     programs (UMaine, community colleges) but has fewer
     apprenticeship pathways.

4. **Client Recommendations**:
   - **Priority 1**: Develop new apprenticeship or certification 
     programs targeting Entry Level/Operators occupations (Pipelayers,
     Highway Maintenance, Operating Engineers, etc.)
   - **Priority 2**: Create bridge programs connecting Skilled Trades
     workers to Management/Engineering career paths
   - **Priority 3**: Establish partnerships between community 
     colleges and AGC member companies for practical training 
     placements

In [ ]:
# Final summary table
summary = {
    'Metric': [
        'Total Companies', 'Total Occupations', 'Total Clusters',
        'Apprenticeship Programs', 'Community College Programs',
        'UMaine Programs', 'Training Gap (Entry Level/Operators)'
    ],
    'Value': [
        len(companies), len(clusters), clusters['cluster_name'].nunique(),
        len(apprentice), len(cc), len(umaine),
        f"{len(clusters[clusters['cluster_name']=='Entry Level/Operators'])} occupations, "
        f"{len(coverage_df[(coverage_df['Cluster']=='Entry Level/Operators') & coverage_df['Has Apprenticeship']])} with training"
    ]
}
summary_df = pd.DataFrame(summary)
print("Project Summary:")
summary_df